# 31. Information Extraction

## Overview
Information extraction converts unstructured text into structured facts, such as people, organizations, locations, dates, and numeric values. This notebook demonstrates a lightweight baseline using NLTK named-entity recognition and regular expressions.

Rule-based patterns are transparent and useful for constrained formats, but they do not replace robust entity recognition systems.

## 1. Import dependencies

The NLTK chunker requires tokenization, POS tagging, and named-entity resources.

In [1]:
import re
import pandas as pd
import nltk

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

from nltk import word_tokenize, pos_tag, ne_chunk

[nltk_data] Downloading package punkt_tab to /home/mg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/mg/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /home/mg/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /home/mg/nltk_data...
[nltk_data]   Package words is already up-to-date!


## 2. Prepare a sample text

In [2]:
text = (
    "Acme Analytics opened an office in Paris on 12 March 2025. "
    "Chief executive Maya Chen said the company invested $2.5 million "
    "in the project. The team will hire 40 engineers by September."
)

print(text)

Acme Analytics opened an office in Paris on 12 March 2025. Chief executive Maya Chen said the company invested $2.5 million in the project. The team will hire 40 engineers by September.


## 3. Extract named entities with NLTK

The chunker identifies likely entity spans from POS-tagged tokens. Its labels and accuracy are limited by the pretrained model.

In [3]:
tokens = word_tokenize(text)
tagged_tokens = pos_tag(tokens)
tree = ne_chunk(tagged_tokens, binary=False)

def extract_entities(chunk_tree):
    records = []
    for node in chunk_tree:
        if hasattr(node, "label"):
            records.append({
                "entity": " ".join(token for token, _ in node.leaves()),
                "entity_type": node.label(),
            })
    return records

entity_df = pd.DataFrame(extract_entities(tree), columns=["entity", "entity_type"])
display(entity_df)

,entity,entity_type
0,Acme,PERSON
1,Analytics,PERSON
2,Paris,GPE
3,Maya Chen,PERSON


## 4. Extract dates and monetary amounts with patterns

Regular expressions work well when text follows predictable formats. They can miss valid variants and may match strings that are not true facts.

In [4]:
date_pattern = re.compile(
    r"\b(?:\d{1,2}\s+)?(?:January|February|March|April|May|June|July|August|September|October|November|December)"
    r"(?:\s+\d{4})?\b",
    flags=re.IGNORECASE,
)
money_pattern = re.compile(r"\$\s?\d+(?:,\d{3})*(?:\.\d+)?")
number_pattern = re.compile(r"\b\d+(?:,\d{3})*\b")

extracted_dates = date_pattern.findall(text)
extracted_money = money_pattern.findall(text)
extracted_numbers = number_pattern.findall(text)

print("Dates:", extracted_dates)
print("Money:", extracted_money)
print("Numbers:", extracted_numbers)

Dates: ['12 March 2025', 'September']
Money: ['$2.5']
Numbers: ['12', '2025', '2', '5', '40']


## 5. Combine extracted values into a structured record

This table demonstrates a compact schema for downstream analysis. The fields are extracted from the example text and are not independently verified.

In [5]:
records = []

for value in extracted_dates:
    records.append({"value": value, "type": "DATE", "method": "regular expression"})

for value in extracted_money:
    records.append({"value": value, "type": "MONEY", "method": "regular expression"})

for value in extracted_numbers:
    records.append({"value": value, "type": "NUMBER", "method": "regular expression"})

structured_df = pd.DataFrame(records, columns=["value", "type", "method"])
display(structured_df)

,value,type,method
0,12 March 2025,DATE,regular expression
1,September,DATE,regular expression
2,$2.5,MONEY,regular expression
3,12,NUMBER,regular expression
4,2025,NUMBER,regular expression
5,2,NUMBER,regular expression
6,5,NUMBER,regular expression
7,40,NUMBER,regular expression


## 6. Extract simple key-value patterns

For controlled text, simple rules can identify phrases such as a company name or a hiring target. These patterns should be adapted to the document template.

In [6]:
hiring_pattern = re.compile(
    r"hire\s+(\d+)\s+([A-Za-z]+)",
    flags=re.IGNORECASE,
)
hiring_match = hiring_pattern.search(text)

if hiring_match:
    hiring_info = {
        "planned_hires": int(hiring_match.group(1)),
        "role": hiring_match.group(2),
    }
else:
    hiring_info = None

print(hiring_info)

{'planned_hires': 40, 'role': 'engineers'}


## Interpretation and limitations

- NER systems predict entity spans and may miss or mislabel names.
- Regular expressions are effective for constrained formats but are brittle when wording changes.
- Extracted facts should retain their source text and be validated when accuracy is important.
- Dates, amounts, and numbers need contextual interpretation before being treated as verified information.

## Summary

This notebook combines NLTK named-entity recognition and regular expressions to turn sample prose into structured entity, date, money, and numeric records.